# How to define custom neural nets

`sbi` allows you to specify a specific density estimator for each of the implemented methods.
We support a variety of density estimators, e.g., mixtures of Gaussians, normalizing
flows, and diffusion models. Some of the density estimators are implemented as part of
`sbi`, for others we rely on other packages like
[`nflows`](https://github.com/bayesiains/nflows/) or [`zuko`](https://github.com/probabilists/zuko). 

For all options, check the API reference
[here](https://sbi.readthedocs.io/en/latest/sbi.html).

## Changing the type of density estimator

The density estimator is chosen by passing a **config object** in the `density_estimator` keyword argument to the inference object (`NPE` or `NLE`). There is one config class per model, so the choice of model is the choice of class: `MAFConfig` for a Masked Autoregressive Flow, `NSFConfig` for a Neural Spline Flow, and so on.

Note that `MAFConfig` or `NSFConfig` correspond to `nflows` density
estimators. Those have proven to work well, but the `nflows` package is not maintained
anymore. To use more recent and actively maintained density estimators, we tentatively
recommend using `zuko`, e.g., `ZukoMAFConfig` or `ZukoNSFConfig`. 

In [ ]:
import torch

from sbi.inference import NPE, NRE
from sbi.utils import BoxUniform

In [ ]:
from sbi.neural_nets import ZukoMAFConfig

prior = BoxUniform(torch.zeros(2), torch.ones(2))
inference = NPE(prior=prior, density_estimator=ZukoMAFConfig())

In the case of `NRE`, the argument is called `classifier`:

In [ ]:
from sbi.neural_nets import ResNetClassifierConfig

inference = NRE(prior=prior, classifier=ResNetClassifierConfig())

## Changing hyperparameters of density estimators

The hyperparameters of a model are the constructor arguments of its config, so you can tune them for the problem at hand.

Here, because we want to use N*P*E, we pass the config to the `density_estimator` argument of `NPE`. In this example, we will create a neural spline flow (`ZukoNSFConfig`) with `60` hidden units and `3` transform layers:

In [ ]:
from sbi.neural_nets import ZukoNSFConfig

density_estimator = ZukoNSFConfig(hidden_features=60, num_transforms=3)
inference = NPE(prior=prior, density_estimator=density_estimator)

A config only accepts the settings its own model has, so a setting that belongs to a different model, or a misspelled one, raises immediately instead of being ignored:

In [ ]:
from sbi.neural_nets import ZukoMAFConfig

try:
    # `num_bins` is a spline setting, and a MAF has no splines.
    ZukoMAFConfig(num_bins=8)
except TypeError as e:
    print(e)

It is also possible to pass an `embedding_net` to a config to automatically
learn summary statistics from high-dimensional simulation outputs. You can find a more
detailed tutorial on this in [04_embedding_networks](https://sbi.readthedocs.io/en/latest/how_to_guide/04_embedding_networks.html).

The full list of configs, together with embedding nets, z-scoring, and how to migrate from the older string and factory-function interfaces, is in [how to configure the neural network](27_estimator_configs.ipynb).

## Building new density estimators from scratch

Finally, it is also possible to implement your own density estimator from scratch, e.g., including embedding nets to preprocess data, or to a density estimator architecture of your choice.

For this, the `density_estimator` argument needs to be a function that takes `theta` and `x` batches as arguments to then construct the density estimator after the first set of simulations was generated. This is what a config does through its `build` method, and what the factory functions in `sbi/neural_nets/factory.py` return.

The returned `density_estimator` object needs to be a subclass of `DensityEstimator`, which requires to implement three methods:
    
- `log_prob(input, condition, **kwargs)`: Return the log probabilities of the inputs given a condition or multiple i.e. batched conditions.
- `loss(input, condition, **kwargs)`: Return the loss for training the density estimator.
- `sample(sample_shape, condition, **kwargs)`: Return samples from the density estimator.

See more information on the [Reference API page](https://sbi.readthedocs.io/en/latest/sbi.html).